In [1]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 11.9 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 123.3 MB/s eta 0:00:01
     |████████████████████████████████| 506 kB 24.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

url = "https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD"
df = pd.read_csv(url, dtype=str)

df.to_csv("../data/raw/restaurant_inspections_raw.csv", index=False)

print(df.shape)
print(df.columns.tolist())
df.head()

(294406, 27)
['CAMIS', 'DBA', 'BORO', 'BUILDING', 'STREET', 'ZIPCODE', 'PHONE', 'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE', 'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE', 'GRADE DATE', 'RECORD DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude', 'Community Board', 'Council District', 'Census Tract', 'BIN', 'BBL', 'NTA', 'Location']


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,40536414,LA CABANA,Brooklyn,106264,FLATBUSH AVENUE,NaN,7182870248,Latin American,03/21/2023,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,0,0,NaN,NaN,NaN,NaN,3,NaN,NaN
1,50089033,THE LEGEND'S LOUNGE,Manhattan,2823,FREDERICK DOUGLASS BOULEVARD,NaN,9143544662,NaN,01/01/1900,NaN,...,NaN,0,0,NaN,NaN,NaN,NaN,1,NaN,NaN
2,41718673,BAREBURGER,Brooklyn,145149,COURT ST,NaN,3475296673,Hamburgers,01/08/2024,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,0,0,NaN,NaN,NaN,NaN,3,NaN,NaN
3,50108113,KAUSA,Manhattan,745,9 AVENUE,10019,6465970651,Peruvian,08/11/2025,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.763686690377,-73.988809245516,104,03,013300,1026685,1010600031,MN15,POINT (-73.988809245516 40.763686690377)
4,41718493,ON TRACK BAR (OTB),Brooklyn,141,BROADWAY,11211,3477631481,New American,12/04/2022,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.710367149637,-73.963543827048,301,34,054900,3063564,3024570043,BK73,POINT (-73.963543827048 40.710367149637)


In [2]:
df.isna().sum()

CAMIS                         0
DBA                           3
BORO                          0
BUILDING                   1010
STREET                       58
ZIPCODE                    3154
PHONE                       110
CUISINE DESCRIPTION        3997
INSPECTION DATE               0
ACTION                     3917
VIOLATION CODE             6983
VIOLATION DESCRIPTION      6983
CRITICAL FLAG                 0
SCORE                     17389
GRADE                    150179
GRADE DATE               160401
RECORD DATE                   0
INSPECTION TYPE            3917
Latitude                   1666
Longitude                  1666
Community Board            4751
Council District           4737
Census Tract               4737
BIN                        5904
BBL                        1666
NTA                        4751
Location                   4751
dtype: int64

In [3]:
print("Total rows:", len(df))
print("Unique restaurants:", df["CAMIS"].nunique())
print("Unique inspections:", df[["CAMIS", "INSPECTION DATE"]].drop_duplicates().shape[0])

Total rows: 294406
Unique restaurants: 31369
Unique inspections: 87721


In [4]:
for col in ["BORO", "GRADE", "CRITICAL FLAG", "INSPECTION TYPE", "ACTION"]:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).head(15))


===== BORO =====
BORO
Manhattan        108989
Brooklyn          74430
Queens            73836
Bronx             27418
Staten Island      9422
0                   311
Name: count, dtype: int64

===== GRADE =====
GRADE
NaN    150179
A       97405
B       17745
C       12821
N       10234
Z        5315
P         707
Name: count, dtype: int64

===== CRITICAL FLAG =====
CRITICAL FLAG
Critical          153991
Not Critical      131621
Not Applicable      8794
Name: count, dtype: int64

===== INSPECTION TYPE =====
INSPECTION TYPE
Cycle Inspection / Initial Inspection                157115
Cycle Inspection / Re-inspection                      56568
Pre-permit (Operational) / Initial Inspection         40234
Pre-permit (Operational) / Re-inspection              10572
Administrative Miscellaneous / Initial Inspection      9604
NaN                                                    3917
Pre-permit (Non-operational) / Initial Inspection      3405
Cycle Inspection / Reopening Inspection            

In [6]:
df["CRITICAL FLAG"].value_counts(dropna=False)

CRITICAL FLAG
Critical          153991
Not Critical      131621
Not Applicable      8794
Name: count, dtype: int64

In [5]:
dates = pd.to_datetime(df["INSPECTION DATE"], errors="coerce")
print(dates.min(), dates.max())
print(dates.dt.year.value_counts().sort_index())

1900-01-01 00:00:00 2026-10-21 00:00:00
INSPECTION DATE
1900     3917
2007       51
2008       29
2009        4
2010       13
2011        6
2012        6
2013        2
2014       12
2015       22
2016      183
2017      364
2018      545
2019      661
2020      157
2021      206
2022     7211
2023    38523
2024    75563
2025    85703
2026    81228
Name: count, dtype: int64


## Business Rule Validation

In [7]:
pd.crosstab(df["ACTION"], df["CRITICAL FLAG"], dropna=False)

CRITICAL FLAG,Critical,Not Applicable,Not Critical
ACTION,,,
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.,7073,5,4002
Establishment re-closed by DOHMH.,674,1,588
Establishment re-opened by DOHMH.,208,298,1112
No violations were recorded at the time of this inspection.,0,2766,186
Violations were cited in the following area(s).,146036,1807,125733
NaN,0,3917,0


In [8]:
# Anomaly 1: "No violations" but flagged as Not Critical
mask_no_viol = df["ACTION"].str.startswith("No violations", na=False) & (df["CRITICAL FLAG"] == "Not Critical")
df.loc[mask_no_viol, ["VIOLATION CODE", "INSPECTION TYPE"]].value_counts().head(10)

VIOLATION CODE  INSPECTION TYPE                                  
19-06           Administrative Miscellaneous / Initial Inspection    27
20-04           Administrative Miscellaneous / Initial Inspection    26
19-06           Administrative Miscellaneous / Re-inspection         19
20-01           Administrative Miscellaneous / Initial Inspection    15
28-03           Administrative Miscellaneous / Initial Inspection    15
28-01           Administrative Miscellaneous / Initial Inspection    14
20-08           Administrative Miscellaneous / Initial Inspection    10
18-11           Administrative Miscellaneous / Initial Inspection     6
28-03           Administrative Miscellaneous / Re-inspection          6
20-04           Administrative Miscellaneous / Re-inspection          6
Name: count, dtype: int64

In [9]:
# Anomaly 2: violations cited but flagged as Not Applicable
mask_na_viol = df["ACTION"].str.startswith("Violations were cited", na=False) & (df["CRITICAL FLAG"] == "Not Applicable")
print(df.loc[mask_na_viol, "INSPECTION TYPE"].value_counts().head(10))
print(df.loc[mask_na_viol, "VIOLATION CODE"].value_counts().head(10))

INSPECTION TYPE
Smoke-Free Air Act / Initial Inspection              620
Calorie Posting / Initial Inspection                 467
Trans Fat / Initial Inspection                       405
Calorie Posting / Re-inspection                       81
Sodium Warning / Initial Inspection                   78
Smoke-Free Air Act / Re-inspection                    63
Trans Fat / Re-inspection                             57
Calorie Posting / Compliance Inspection               11
Administrative Miscellaneous / Initial Inspection      7
Smoke-Free Air Act / Compliance Inspection             6
Name: count, dtype: int64
VIOLATION CODE
16-02    448
16-03    286
15-27    148
15-21    139
16-04    139
16-06    134
15-37    102
15-42     93
15-39     56
15-01     51
Name: count, dtype: int64


In [10]:
# Convert SCORE to numeric for comparison
df["SCORE_NUM"] = pd.to_numeric(df["SCORE"], errors="coerce")

# Rule check 1: score range for each grade
df.groupby("GRADE", dropna=False)["SCORE_NUM"].agg(["count", "min", "max"])

,count,min,max
GRADE,,,
A,97403,0.0,27.0
B,17745,0.0,66.0
C,12819,0.0,154.0
N,10223,0.0,214.0
P,707,0.0,23.0
Z,5315,0.0,108.0
NaN,132805,0.0,212.0


In [11]:
# Rule check 2: initial inspections - score <= 13 should get A, score >= 14 should have no grade
initial = df[df["INSPECTION TYPE"].str.contains("Initial Inspection", na=False)].copy()
initial["SCORE_BAND"] = pd.cut(
    initial["SCORE_NUM"],
    bins=[-1, 13, 27, 1000],
    labels=["0-13", "14-27", "28+"]
)
pd.crosstab(initial["SCORE_BAND"], initial["GRADE"], dropna=False)

GRADE,A,N
SCORE_BAND,,
NaN,2,11
0-13,68363,500
14-27,0,4270
28+,0,5003


In [12]:
# Step 1: collapse to inspection level (one row per inspection)
insp = df.drop_duplicates(subset=["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]).copy()

# Step 2: keep only in-scope programs
in_scope_programs = ["Cycle Inspection", "Pre-permit (Operational)"]
insp["PROGRAM"] = insp["INSPECTION TYPE"].str.split(" / ").str[0]
insp["TYPE"] = insp["INSPECTION TYPE"].str.split(" / ").str[1]
insp = insp[insp["PROGRAM"].isin(in_scope_programs)]

# Step 3: make missing grades visible
insp["GRADE_FILLED"] = insp["GRADE"].fillna("No grade")

print("In-scope inspections:", len(insp))

In-scope inspections: 79834


In [13]:
# Rule check 2 (revised): initial inspections, score band vs grade
initial = insp[insp["TYPE"] == "Initial Inspection"].copy()
initial["SCORE_BAND"] = pd.cut(initial["SCORE_NUM"], bins=[-1, 13, 27, 1000], labels=["0-13", "14-27", "28+"])
pd.crosstab(initial["SCORE_BAND"], initial["GRADE_FILLED"])

GRADE_FILLED,A,N,No grade
SCORE_BAND,,,
0-13,31704,61,1206
14-27,0,951,11356
28+,0,720,10874


In [14]:
# Rule check 1 (revised): how many A/B/C grades do NOT match the score?
abc = insp[insp["GRADE"].isin(["A", "B", "C"])].copy()
abc["EXPECTED_GRADE"] = pd.cut(abc["SCORE_NUM"], bins=[-1, 13, 27, 1000], labels=["A", "B", "C"]).astype(str)
abc["MATCH"] = abc["GRADE"] == abc["EXPECTED_GRADE"]

print(abc["MATCH"].value_counts())
print(abc["MATCH"].value_counts(normalize=True).round(3))
pd.crosstab(abc["GRADE"], abc["EXPECTED_GRADE"])

MATCH
True     50600
False      271
Name: count, dtype: int64
MATCH
True     0.995
False    0.005
Name: proportion, dtype: float64


EXPECTED_GRADE,A,B,C
GRADE,,,
A,44411,1,0
B,10,4234,14
C,240,6,1955
